# Part 0: Imports and filesystem mounting

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import os
print(f"CPU count: {os.cpu_count()}")
import multiprocessing
print(f"Available cores: {multiprocessing.cpu_count()}")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

project_dir = '/content/drive/My Drive/Colab Notebooks/rule_learning_sbi/'
os.chdir(project_dir)

root_path = Path(project_dir) / 'storage'
data_path = root_path / 'data'
save_path = root_path / 'model_identification'
make_plots = False

# Check and install TensorFlow if needed (works for both TPU and GPU)
import sys
# Workaround for ValueError: Name tf.RaggedTensorSpec has already been registered
# This can happen in Colab due to TensorFlow being partially loaded before explicit import.
modules_to_remove = [k for k in list(sys.modules.keys()) if 'tensorflow' in k.lower()]
for mod in modules_to_remove:
    del sys.modules[mod]

# install
!pip install tensorflow -q

# model specification
import helpers
import importlib
importlib.reload(helpers.params)
from helpers.params import Param, ParamConfig

RULEQHYP_MODEL_NAME = 'ruleQhyp'
RULEQHYP_BASE_MODEL_SPEC = ParamConfig([
    Param(name='Ap', lb=0.0, ub=1.0), # base learning rate
    Param(name='An', lb=0.0, ub=1.0, default='Ap'), # learning rate multiplier for neg rpe
    Param(name='Agcf', lb=0.0, ub=1.0, default=0.0), # learning rate multiplier for CF (unchosen) updates
    Param(name='B', lb=0.5, ub=10.0), # inv temp
    Param(name='E', lb=0.0, ub=0.25, default=0.0), # noise
    Param(name='Ft', lb=0.0, ub=0.2, default=0.0), # between-trial forgetting
    Param(name='Fb', lb=0.0, ub=0.5, default=0.0), # between-block forgetting
    Param(name='Ex', lb=0.25, ub=1.0, default=1.0), # exploit threshold
    Param(name='L', lb=0.0, ub=0.5, default=0.0), # lapse rate
    Param(name='R', lb=0.25, ub=1.0, default=0.0), # recovery rate
    Param(name='Ba', lb=-2.5, ub=2.5, default=0.0), # side (action) bias
    Param(name='Ws', lb=0.0, ub=2.5, default=0.0), # win-stay weight (contributes to heuristic when attentive)
    Param(name='Ls', lb=0.0, ub=2.5, default='Ws'), # lose-shift weight (contributes to heuristic when attentive)
    Param(name='Sa', lb=-2.5, ub=2.5, default=0.0), # sticky action (inattentive)
    Param(name='Sda', lb=0.0, ub=1.0, default=1.0), # sticky action decay (inattentive)
    Param(name='Sr', lb=-2.5, ub=2.5, default=0.0), # sticky rule (attentive)
    Param(name='Sdr', lb=0.0, ub=1.0, default=1.0), # sticky rule decay (attentive)
    Param(name='Dle', lb=0.0, ub=1.0, default=1.0/7), # default Q-value go left
    Param(name='Dri', lb=0.0, ub=1.0, default=1.0/7), # default Q-value go right
    Param(name='Dli', lb=0.0, ub=1.0, default=1.0/7), # default Q-value go light
    Param(name='Dda', lb=0.0, ub=1.0, default=1.0/7), # default Q-value go dark
    Param(name='Dcl', lb=0.0, ub=1.0, default=1.0/7), # default Q-value go click
    Param(name='Dsi', lb=0.0, ub=1.0, default=1.0/7), # default Q-value go silent
    Param(name='Dal', lb=0.0, ub=1.0, default=1.0/7) # default Q-value alternate
],
base_name=RULEQHYP_MODEL_NAME)

# dictionary specifying all possible variants (that we're interested in) of rule-Q learning model
# Each top-level feature can be independently enabled/disabled
# Structure: {feature_name: {option_value: {sub_options...}}}
# None = feature is free (enabled), other values = fixed to that value (disabled with that default)
# Empty dict {} = leaf node (no further options)
#
# Each top-level feature (An, Agcf, E, Ft, Fb, Ex, LR, WsLs, Sr)
# can be turned on/off independently of the others

RULEQHYP_FEATURE_OPTIONS = {
    # An: Separate negative learning rate
    # Can be free (None) or linked to Ap (default behavior)
    'An': {
        None: {},  # Free An (independent of Ap) - FEATURE ON
        'Ap': {}   # An=Ap (default, disabled) - FEATURE OFF
    },

    # E: Random decision noise (epsilon)
    # Can be free (None) or fixed at 0 (default, disabled)
    # If free, can have Ba (side bias) and Sa (sticky action) as sub-features
    'E': {
        None: {  # E is free - FEATURE ON
            # Ba: Side bias (requires E)
            'Ba': {
                None: {},  # Free Ba
                0.0: {}    # Ba=0 (disabled)
            },
            # Sa: Sticky action (requires E for heuristic policy)
            'Sa': {
                None: {  # Free Sa
                    # Sda: Sticky action decay
                    'Sda': {
                        None: {},  # Free Sda
                        1.0: {}    # Sda=1 (immediate decay)
                    }
                },
                0.0: {}  # Sa=0 (disabled)
            }
        },
        0.0: {}  # E=0 (default, disabled) - FEATURE OFF - Ba and Sa cannot be enabled if E=0
    },

    # Ft: Trial forgetting
    # Can be free (None) or fixed at 0 (default, disabled)
    'Ft': {
        None: {},  # Free Ft - FEATURE ON
        0.0: {}    # Ft=0 (default, disabled) - FEATURE OFF
    },

    # Fb: Block forgetting
    # Can be free (None) or fixed at 0 (default, disabled)
    'Fb': {
        None: {},  # Free Fb - FEATURE ON
        0.0: {}    # Fb=0 (default, disabled) - FEATURE OFF
    },

    # Ex: Exploit threshold
    # Can be free (None) or fixed at 1.0 (default, disabled - always explore)
    'Ex': {
        None: {},  # Free Ex - FEATURE ON
        1.0: {}    # Ex=1.0 (default, disabled - always explore) - FEATURE OFF
    },

    # LR: Lapse and Recovery (attention dynamics)
    # Only two settings:
    # - 'both': fit both L and R
    # - 'disabled': fix both to 0
    'LR': {
        'on': {  # FEATURE ON
            'L': {None: {}},
            'R': {None: {}}
        },
        'off': {  # FEATURE OFF (default)
            'L': {0.0: {}},
            'R': {1.0: {}}
        }
    },

    # WsLs: Win-stay/Lose-shift heuristic
    # Can be linked (Ls=Ws), separate, Ws only, Ls only, or both disabled
    'WsLs': {
        'linked': {  # Ls=Ws (1 free param) - FEATURE ON
            'Ws': {None: {}},
            'Ls': {'Ws': {}}  # Ls references Ws
        },
        'separate': {  # Both Ws and Ls free (2 free params) - FEATURE ON
            'Ws': {None: {}},
            'Ls': {None: {}}
        },
        'Ws_only': {  # Only Ws free, Ls=0 - FEATURE PARTIALLY ON
            'Ws': {None: {}},
            'Ls': {0.0: {}}
        },
        'Ls_only': {  # Only Ls free, Ws=0 - FEATURE PARTIALLY ON
            'Ws': {0.0: {}},
            'Ls': {None: {}}
        },
        'disabled': {  # Both Ws=0 and Ls=0 (default) - FEATURE OFF
            'Ws': {0.0: {}},
            'Ls': {0.0: {}}
        }
    },

    # Sr: Sticky rule (attentive)
    # Can be full (Sr and Sdr free), Sr only (Sdr=0), or disabled
    'Sr': {
        'full': {  # Both Sr and Sdr free - FEATURE ON
            'Sr': {None: {}},
            'Sdr': {None: {}}
        },
        'Sr_only': {  # Only Sr free, Sdr=0 - FEATURE PARTIALLY ON
            'Sr': {None: {}},
            'Sdr': {1.0: {}} # immediate decay
        },
        'disabled': {  # Both Sr=0 and Sdr=0 (default) - FEATURE OFF
            'Sr': {0.0: {}},
            'Sdr': {1.0: {}}
        }
    }
}

# Base features that are always free (not in the tree)
# These are the minimal model: Ap and B
RULEQHYP_BASE_FEATURES = ['Ap', 'B']

# Default Q-values are always uniform (1/7), so not in the tree
# Dle, Dri, Dli, Dda, Dcl, Dsi, Dal all default to 1.0/7.0


# ============================================================================
# QLEARN MODEL FEATURE OPTIONS
# ============================================================================
# Similar tree structure for Qlearn model (direct state-action Q-learning)
# Qlearn has fewer features than ruleQhyp (no rule selection features)

QLEARN_MODEL_NAME = 'Qlearn'
QLEARN_BASE_MODEL_SPEC = ParamConfig([
    Param(name='Ap', lb=0.01, ub=1.0), # base learning rate
    Param(name='An', lb=0.01, ub=1.0, default='Ap'), # learning rate multiplier for neg rpe
    Param(name='Agcf', lb=0.0, ub=1.0, default=0.0), # learning rate multiplier for CF (unchosen) updates
    Param(name='B', lb=0.5, ub=10.0), # inv temp
    Param(name='E', lb=0.0, ub=0.25, default=0.0), # noise
    Param(name='Ft', lb=0.0, ub=0.2, default=0.0), # between-trial forgetting
    Param(name='Fb', lb=0.0, ub=0.5, default=0.0), # between-block forgetting
    Param(name='L', lb=0.0, ub=0.5, default=0.0), # lapse rate
    Param(name='R', lb=0.25, ub=1.0, default=0.0), # recovery rate
    Param(name='Ba', lb=-2.5, ub=2.5, default=0.0), # side (action) bias
    Param(name='Ws', lb=0.0, ub=2.5, default=0.0), # win-stay weight (contributes to heuristic when attentive)
    Param(name='Ls', lb=0.0, ub=2.5, default='Ws'), # lose-shift weight (contributes to heuristic when attentive)
    Param(name='Sa', lb=-2.5, ub=2.5, default=0.0), # sticky action (inattentive)
    Param(name='Sda', lb=0.0, ub=1.0, default=1.0), # sticky action decay (inattentive)
    Param(name='Dq', lb=0.0, ub=1.0, default=0.5) # default Q-value
],
base_name=QLEARN_MODEL_NAME)

QLEARN_FEATURE_OPTIONS = {
    # An: Separate negative learning rate
    # Can be free (None) or linked to Ap (default behavior)
    'An': {
        None: {},  # Free An (independent of Ap) - FEATURE ON
        'Ap': {}   # An=Ap (default, disabled) - FEATURE OFF
    },

    # Agcf: Counterfactual learning rate multiplier
    # Can be free (None) or fixed at 0 (default, disabled)
    'Agcf': {
        None: {},  # Free Agcf - FEATURE ON
        0.0: {}    # Agcf=0 (default, disabled) - FEATURE OFF
    },

    # E: Random decision noise (epsilon)
    # Can be free (None) or fixed at 0 (default, disabled)
    # If free, can have Ba (side bias) and Sa (sticky action) as sub-features
    'E': {
        None: {  # E is free - FEATURE ON
            # Ba: Side bias (requires E)
            'Ba': {
                None: {},  # Free Ba
                0.0: {}    # Ba=0 (disabled)
            },
            # Sa: Sticky action (requires E for heuristic policy)
            'Sa': {
                None: {  # Free Sa
                    # Sda: Sticky action decay
                    'Sda': {
                        None: {},  # Free Sda
                        1.0: {}    # Sda=1 (immediate decay)
                    }
                },
                0.0: {}  # Sa=0 (disabled)
            }
        },
        0.0: {}  # E=0 (default, disabled) - FEATURE OFF - Ba and Sa cannot be enabled if E=0
    },

    # Ft: Trial forgetting
    # Can be free (None) or fixed at 0 (default, disabled)
    'Ft': {
        None: {},  # Free Ft - FEATURE ON
        0.0: {}    # Ft=0 (default, disabled) - FEATURE OFF
    },

    # Fb: Block forgetting
    # Can be free (None) or fixed at 0 (default, disabled)
    'Fb': {
        None: {},  # Free Fb - FEATURE ON
        0.0: {}    # Fb=0 (default, disabled) - FEATURE OFF
    },

    # LR: Lapse and Recovery (attention dynamics)
    # Only two settings:
    # - 'on': fit both L and R
    # - 'off': fix both to 0
    'LR': {
        'on': {  # FEATURE ON
            'L': {None: {}},
            'R': {None: {}}
        },
        'off': {  # FEATURE OFF (default)
            'L': {0.0: {}},
            'R': {1.0: {}}
        }
    },

    # WsLs: Win-stay/Lose-shift heuristic (action-level)
    # Can be linked (Ls=Ws), separate, Ws only, Ls only, or both disabled
    'WsLs': {
        'linked': {  # Ls=Ws (1 free param) - FEATURE ON
            'Ws': {None: {}},
            'Ls': {'Ws': {}}  # Ls references Ws
        },
        'separate': {  # Both Ws and Ls free (2 free params) - FEATURE ON
            'Ws': {None: {}},
            'Ls': {None: {}}
        },
        'Ws_only': {  # Only Ws free, Ls=0 - FEATURE PARTIALLY ON
            'Ws': {None: {}},
            'Ls': {0.0: {}}
        },
        'Ls_only': {  # Only Ls free, Ws=0 - FEATURE PARTIALLY ON
            'Ws': {0.0: {}},
            'Ls': {None: {}}
        },
        'disabled': {  # Both Ws=0 and Ls=0 (default) - FEATURE OFF
            'Ws': {0.0: {}},
            'Ls': {0.0: {}}
        }
    }
}

# Base features that are always free (not in the tree)
# These are the minimal model: Ap and B
QLEARN_BASE_FEATURES = ['Ap', 'B']
# Dq (default Q-value) is always uniform initialization, so not in the tree
# It defaults to a uniform value (typically 0.5 for binary actions)

MODEL_NAMES = [QLEARN_MODEL_NAME, RULEQHYP_MODEL_NAME]


CPU count: 8
Available cores: 8
Mounted at /content/drive


# Part 1: Simulating Data


In [ ]:
import sys
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import importlib
from pathlib import Path

# Import params and reload to get latest serialize() method
from helpers import params
importlib.reload(params)
from helpers.params import ParamConfig

# Import simulation package and submodules first
import simulation
from simulation import generative_models
from simulation import simulate
from simulation import constants
from simulation import utils

# Reload modules
importlib.reload(simulation.generative_models.ruleQhyp)
importlib.reload(simulation.generative_models.Qlearn)
importlib.reload(simulation.simulate)
importlib.reload(simulation.constants)

# Import simulation functions
from simulation.simulate import simulate_all_agents_parallel
from simulation.generative_models.ruleQhyp import generate as generate_ruleQhyp
from simulation.generative_models.Qlearn import generate as generate_Qlearn
from simulation.utils import generate_variants_from_tree

from multiprocessing import cpu_count

# Settings
N_TOTAL_PER_BASE = 50000
MAX_RETRIES = 25
MAX_SESSIONS_PER_RULE = 9
MIN_PERFORMANCE = 0.25
BATCH_SIZE = 1000
RESTART = False
N_WORKERS = cpu_count() - 1  # Leave one core free for system processes

print("="*70)
print("PARALLEL SIMULATION WITH INCREMENTAL SAVES")
print("="*70)
print(f"Target: {N_TOTAL_PER_BASE:,} agents per model")
print(f"Parallel workers: {N_WORKERS}")
print(f"Batch size: {BATCH_SIZE:,} agents")
print(f"Restart from scratch: {RESTART}")
print(f"Quality filters:")
print(f"  - Minimum accuracy: {MIN_PERFORMANCE*100:.0f}%")
print(f"  - Max sessions per rule: {MAX_SESSIONS_PER_RULE}")
print(f"  - Max retries per agent: {MAX_RETRIES}")
print("="*70)

# Generate variants
print("\nGenerating variants from feature trees...")
ruleQhyp_variants = generate_variants_from_tree(RULEQHYP_FEATURE_OPTIONS, RULEQHYP_MODEL_NAME)
qlearn_variants = generate_variants_from_tree(QLEARN_FEATURE_OPTIONS, QLEARN_MODEL_NAME)
print(f"Found {len(ruleQhyp_variants)} ruleQhyp variants")
print(f"Found {len(qlearn_variants)} Qlearn variants")

# Simulate both models
for base_name, gen_func, base_spec, variants in [
    ("ruleQhyp", generate_ruleQhyp, RULEQHYP_BASE_MODEL_SPEC, ruleQhyp_variants),
    ("Qlearn", generate_Qlearn, QLEARN_BASE_MODEL_SPEC, qlearn_variants),
]:
    print(f"\n{'='*70}")
    print(f"SIMULATING {base_name.upper()}")
    print(f"Variants: {len(variants)}")
    print(f"Workers: {N_WORKERS}")
    print(f"{'='*70}")

    # Use high-level function - handles everything: parallelization, checkpointing,
    # batch saving, merging, and cleanup
    n_agents, n_trial_rows = simulate_all_agents_parallel(
        base_name=base_name,
        gen_func=gen_func,
        base_spec=base_spec,
        variants=variants,
        save_dir=data_path / base_name,
        n_total_per_base=N_TOTAL_PER_BASE,
        max_retries=MAX_RETRIES,
        max_sessions_per_rule=MAX_SESSIONS_PER_RULE,
        min_performance=MIN_PERFORMANCE,
        batch_size=BATCH_SIZE,
        n_workers=N_WORKERS,
        restart=RESTART,
        cleanup_batches=False  # Keep batch files for safety (can manually delete later)
    )

    print(f"\n  Completed: {n_agents:,} agents, {n_trial_rows:,} trial rows")

print("\n" + "="*70)
print("SIMULATION COMPLETE!")
print("="*70)

PARALLEL SIMULATION WITH INCREMENTAL SAVES
Target: 50,000 agents per model
Parallel workers: 7
Batch size: 1,000 agents
Restart from scratch: False
Quality filters:
  - Minimum accuracy: 25%
  - Max sessions per rule: 9
  - Max retries per agent: 25

Generating variants from feature trees...
Found 3360 ruleQhyp variants
Found 1120 Qlearn variants

SIMULATING RULEQHYP
Variants: 3360
Workers: 7
  Distribution: 3360 variants, 14 base agents/variant, 2960 variants get +1 agent (expected total: 50,000)
  No checkpoint found: Starting fresh
  Saving every ~71 variants (~994 agents)

  Starting parallel simulation...

  SAVED BATCH 0: 1,005 agents | Total: 1,005/50,000
  SAVED BATCH 1: 1,005 agents | Total: 2,010/50,000
  SAVED BATCH 2: 1,005 agents | Total: 3,015/50,000
  SAVED BATCH 3: 1,005 agents | Total: 4,020/50,000
  SAVED BATCH 4: 1,005 agents | Total: 5,025/50,000
  SAVED BATCH 5: 1,005 agents | Total: 6,030/50,000
  SAVED BATCH 6: 1,005 agents | Total: 7,035/50,000
  SAVED BATCH 7: 

# Part 2: Feature extraction


In [ ]:
# ============================================================================
# EXTRACT FEATURES AND LABELS FOR RNN MODEL CLASSIFICATION
# Binary classification: ruleQhyp vs Qlearn
# Features: new_session + visual + auditory + action + reward + prev_action +
#   prev_reward  (11 float columns; same layout as real .npy / seqidRNN)
# Architecture: Single-level RNN with state carry-over between sessions
#   - Up to (MAX_SESSIONS-1) sessions × MAX_TRIALS_PER_SESSION trials per agent
# ============================================================================

import sys
import importlib
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import numpy as np
from pathlib import Path

# Output directory for final features
features_dir = save_path / 'rnn_features_model_identification'

# ============================================================================
# CHECK IF FINAL OUTPUT FILES ALREADY EXIST (skip everything if so)
# ============================================================================
expected_files = [
    features_dir / "features_train.npy",
    features_dir / "features_val.npy",
    features_dir / "features_test.npy",
    features_dir / "labels_train.npy",
    features_dir / "labels_val.npy",
    features_dir / "labels_test.npy",
    features_dir / "split_info.pkl"
]

if all(f.exists() for f in expected_files):
    print("="*70)
    print("SKIPPING: Final output files already exist")
    print("="*70)

    train_mmap = np.load(features_dir / "features_train.npy", mmap_mode='r')
    val_mmap = np.load(features_dir / "features_val.npy", mmap_mode='r')
    test_mmap = np.load(features_dir / "features_test.npy", mmap_mode='r')

    print(f"  ✓ Train: {len(train_mmap):,} agents")
    print(f"  ✓ Val:   {len(val_mmap):,} agents")
    print(f"  ✓ Test:  {len(test_mmap):,} agents")
    print(f"  Feature shape: {train_mmap.shape[1:]}")

    del train_mmap, val_mmap, test_mmap

    import pickle
    with open(features_dir / "split_info.pkl", 'rb') as f:
        split_info = pickle.load(f)
    split_indices = split_info.get('split_indices', None)

    saved_paths = {
        'train': {'features': features_dir / "features_train.npy", 'labels': features_dir / "labels_train.npy"},
        'val': {'features': features_dir / "features_val.npy", 'labels': features_dir / "labels_val.npy"},
        'test': {'features': features_dir / "features_test.npy", 'labels': features_dir / "labels_test.npy"}
    }

    print(f"\n  To reprocess, delete: {features_dir}")

else:
    # ============================================================================
    # PROCESS (only if final files don't exist)
    # ============================================================================
    import helpers
    from helpers import io
    from helpers import data_prep
    importlib.reload(helpers.data_prep)
    from helpers.data_prep import (
        extract_and_curate_labels,
        get_features_from_dataframe,
        stratify_indices_from_metadata,
        write_train_val_test_from_chunks,
    )
    import simulation
    from simulation import constants
    importlib.reload(simulation.constants)
    from simulation.constants import MAX_SESSIONS, MAX_TRIALS_PER_SESSION

    max_sessions_grid = MAX_SESSIONS - 1
    chunk_size = 1000

    FEATURE_NAMES = [
        "new_session",
        "visual",
        "auditory",
        "action",
        "reward",
        "prev_action",
        "prev_reward",
    ]

    print("="*70)
    print("EXTRACTING FEATURES AND LABELS FOR STATE CARRY-OVER RNN")
    print("Binary: ruleQhyp vs Qlearn")
    print(f"  feature_names: {FEATURE_NAMES}")
    print("="*70)

    labels_list = []
    chunks_dirs_list = []
    all_metadata_list = []

    base_models = ['ruleQhyp', 'Qlearn']
    for base_model in base_models:
        print(f"\n{'='*70}")
        print(f"Processing {base_model}...")
        print('='*70)

        parquet_path = data_path / base_model / "sim_data.parquet"
        metadata_path = data_path / base_model / "sim_metadata.pkl"
        features_chunks_dir = save_path / f'rnn_features_chunks_{base_model}'

        print(f"\nLoading metadata...")
        all_metadata = io.load_pickle(metadata_path)
        print(f"  Loaded {len(all_metadata):,} agents")
        all_metadata_list.append(all_metadata)

        print(f"\nExtracting and curating labels...")
        labels = extract_and_curate_labels(
            parquet_path=parquet_path,
            all_metadata=all_metadata,
            output_head_config={
                'model': {
                    'label_key_metadata': 'base_model',
                    'category_names': MODEL_NAMES
                }
            },
            chunk_size=chunk_size,
            max_sessions=max_sessions_grid,
            max_trials_per_session=MAX_TRIALS_PER_SESSION,
        )
        labels_list.append(labels)

        print(f"\nExtracting features...")
        _ = get_features_from_dataframe(
            parquet_path=parquet_path,
            all_metadata=all_metadata,
            feature_names=FEATURE_NAMES,
            max_sessions=max_sessions_grid,
            max_trials_per_session=MAX_TRIALS_PER_SESSION,
            chunk_size=chunk_size,
            save_dir=features_chunks_dir
        )
        chunks_dirs_list.append(features_chunks_dir)

    stratify_by = stratify_indices_from_metadata(
        [
            (all_metadata_list[0], "ruleQhyp"),
            (all_metadata_list[1], "Qlearn"),
        ],
        key="model_id",
    )
    saved_paths, split_indices = write_train_val_test_from_chunks(
        chunks_dirs=chunks_dirs_list,
        labels_list=labels_list,
        save_dir=features_dir,
        split_train_val_test=True,
        test_size=0.1,
        val_size=0.1,
        random_state=42,
        stratify_by=stratify_by,
    )

print(f"\nFeatures ready at: {features_dir}")

SKIPPING: Final output files already exist
  ✓ Train: 80,000 agents
  ✓ Val:   10,000 agents
  ✓ Test:  10,000 agents
  Feature shape: (35, 300, 11)

  To reprocess, delete: /content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/rnn_features_model_identification

✓ Features ready at: /content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/rnn_features_model_identification


In [ ]:
import numpy as np
import importlib, helpers
importlib.reload(helpers)
import helpers.inspect_samples
importlib.reload(helpers.inspect_samples)
from helpers import build_index_plan, print_feature_samples, print_label_samples

# inspect simulated data
# --- set your model-ID (seqid/modelID RNN) paths here ---
features_path = "/content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/rnn_features_model_identification/features_train.npy"
labels_path   = "/content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/rnn_features_model_identification/labels_train.npy"  # or .npz

features = np.load(features_path, mmap_mode="r")
N, S, T = int(features.shape[0]), int(features.shape[1]), int(features.shape[2])

plan = build_index_plan(N=N, S=S, T=T, n_agents=3, seed=42)

print_feature_samples("features_train", features, plan)

if labels_path.endswith(".npz"):
    with np.load(labels_path, allow_pickle=True) as z:
        for k in z.files:
            print_label_samples(k, z[k], plan)
else:
    labels = np.load(labels_path, mmap_mode="r")
    print_label_samples("labels_train", labels, plan)


[features_train]
  shape: (80000, 35, 300, 11) dtype: float32
  agent 52365
    session 0
      trial 0: [1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0.]
      trial 1: [0. 1. 0. 1. 0. 1. 0. 0. 1. 0. 0.]
      trial 25: [0. 1. 0. 0. 1. 1. 0. 0. 1. 0. 1.]
    session 1
      trial 0: [1. 1. 0. 0. 1. 0. 1. 1. 0. 0. 0.]
      trial 1: [0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 1.]
      trial 209: [0. 0. 1. 0. 1. 1. 0. 1. 1. 0. 0.]
    session 30
      trial 0: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
      trial 1: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
      trial 60: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
  agent 7139
    session 0
      trial 0: [1. 1. 0. 0. 1. 0. 1. 1. 0. 0. 0.]
      trial 1: [0. 0. 1. 1. 0. 0. 1. 0. 0. 1. 1.]
      trial 157: [0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1.]
    session 1
      trial 0: [1. 1. 0. 1. 0. 0. 1. 1. 0. 0. 0.]
      trial 1: [0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1.]
      trial 292: [0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 1.]
    session 3
      trial 0: [1. 0. 1. 0. 1. 1. 

# Part 3. Hyperparam tuning


### Hyperparam tuning stage 1

In [ ]:
# ============================================================================
# STAGE 1: QUALITATIVE ARCHITECTURE (cell type × uni/bi × num hidden layers)
# ============================================================================

import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import importlib
import helpers.hyperparameter_search.hyperparam_sweep_seqidRNN as hp_sweep
importlib.reload(hp_sweep)

from helpers.hyperparameter_search.hyperparam_sweep_seqidRNN import (
    build_configs, run_config, config_tag,
    print_results, save_results, top_configs,
)

from helpers.rnn.seqidRNN import seqidRNN, train

# ---------------------------------------------------------------------------
# Definitions / configs
# ---------------------------------------------------------------------------
features_dir = save_path / 'rnn_features_model_identification'
results_dir_base = save_path / 'hyperparam_search' /'hyperparam_search_seqid_3stage'
HYPSEARCH_CONFIG_STAGE1 = {
    # Backbone sweep
    "rnn_type": ["gru", "lstm"],
    "bidirectional": [False, True],
    "hidden_size": [64, 128, 256],
    # Fixed head (neutral/default)
    "n_classifier_layers": 1,
    "fc_width_mult": (0.5,),      # one value because n_classifier_layers=1
    "dropout": (0.2, 0.1),
    # Fixed training hparams
    "lr": 1e-3,
    "patience": 15,
    "reduce_lr_patience": 7,
}

STAGE_1_SETTINGS = {
    'train_size': 25000,
    'val_size': 5000,
    'epochs': 50,
    'batch_size': 32,
    'top_k': 3,
}

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
print("\nLoading data...")

X_train = np.load(features_dir / "features_train.npy", mmap_mode='r')
X_val = np.load(features_dir / "features_val.npy", mmap_mode='r')
y_train = np.load(features_dir / "labels_train.npy")
y_val = np.load(features_dir / "labels_val.npy")

print(f"  Train: {X_train.shape}, {y_train.shape}")
print(f"  Val:   {X_val.shape}, {y_val.shape}")

INPUT_SIZE = X_train.shape[-1]
N_MODELS = y_train.shape[1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Prepare data subset
# ---------------------------------------------------------------------------
print("\nSubsetting data...")
rng = np.random.RandomState(42)

train_idx = rng.choice(len(y_train), size=STAGE_1_SETTINGS['train_size'], replace=False)
val_idx = rng.choice(len(y_val), size=STAGE_1_SETTINGS['val_size'], replace=False)

sessions_train = torch.tensor(np.array(X_train[train_idx]), dtype=torch.float32)
sessions_val   = torch.tensor(np.array(X_val[val_idx]),     dtype=torch.float32)
labels_train   = torch.tensor(y_train[train_idx].argmax(axis=1), dtype=torch.long)
labels_val     = torch.tensor(y_val[val_idx].argmax(axis=1),     dtype=torch.long)

n_real_train = (sessions_train[:, :, 0, 0] != -1).sum(dim=1).long()
n_real_val   = (sessions_val[:, :, 0, 0] != -1).sum(dim=1).long()

print(f"  Train: {sessions_train.shape}, real sessions: {n_real_train.min()}-{n_real_train.max()}")
print(f"  Val:   {sessions_val.shape}, real sessions: {n_real_val.min()}-{n_real_val.max()}")
print(f"  Device: {DEVICE}")

BS = STAGE_1_SETTINGS['batch_size']
train_loader = DataLoader(TensorDataset(sessions_train, labels_train, n_real_train), batch_size=BS, shuffle=True)
val_loader   = DataLoader(TensorDataset(sessions_val, labels_val, n_real_val), batch_size=BS)


# ---------------------------------------------------------------------------
# Run Stage 1
# ---------------------------------------------------------------------------
configs = build_configs(HYPSEARCH_CONFIG_STAGE1)

print("\n" + "=" * 70)
print(f"STAGE 1: QUALITATIVE ARCHITECTURE — {len(configs)} configs")
print("=" * 70)

SAVE_DIR = results_dir_base / "stage1"

stage1_results = []
for i, cfg in enumerate(configs):
    print(f"\n--- Config {i+1}/{len(configs)}: {config_tag(cfg)} ---")
    result = run_config(
        cfg, train_loader, val_loader,
        INPUT_SIZE, N_MODELS, DEVICE,
        n_epochs=STAGE_1_SETTINGS['epochs'],
        save_dir=SAVE_DIR,
    )
    stage1_results.append(result)

# Final ranked summary overwrites the append-order JSON
save_results(stage1_results, SAVE_DIR)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print_results(stage1_results, "STAGE 1 RESULTS")

top_stage1 = top_configs(stage1_results, k=STAGE_1_SETTINGS['top_k'])
print(f"\nCarrying top {STAGE_1_SETTINGS['top_k']} to Stage 2:")
for cfg in top_stage1:
    print(f"  {config_tag(cfg)}")


Loading data...
  Train: (80000, 35, 300, 11), (80000, 2)
  Val:   (10000, 35, 300, 11), (10000, 2)

Subsetting data...
  Train: torch.Size([25000, 35, 300, 11]), real sessions: 3-35
  Val:   torch.Size([5000, 35, 300, 11]), real sessions: 3-35
  Device: cuda

STAGE 1: QUALITATIVE ARCHITECTURE — 12 configs

--- Config 1/12: gru_uni_64u_fc1_m0.5_d0.2-0.1 ---
    [SKIP] Found gru_uni_64u_fc1_m0.5_d0.2-0.1.pt, loading cached result

--- Config 2/12: gru_uni_128u_fc1_m0.5_d0.2-0.1 ---
    [SKIP] Found gru_uni_128u_fc1_m0.5_d0.2-0.1.pt, loading cached result

--- Config 3/12: gru_uni_256u_fc1_m0.5_d0.2-0.1 ---
    [SKIP] Found gru_uni_256u_fc1_m0.5_d0.2-0.1.pt, loading cached result

--- Config 4/12: gru_bi_64u_fc1_m0.5_d0.2-0.1 ---
    [SKIP] Found gru_bi_64u_fc1_m0.5_d0.2-0.1.pt, loading cached result

--- Config 5/12: gru_bi_128u_fc1_m0.5_d0.2-0.1 ---
    [SKIP] Found gru_bi_128u_fc1_m0.5_d0.2-0.1.pt, loading cached result

--- Config 6/12: gru_bi_256u_fc1_m0.5_d0.2-0.1 ---


Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

### Hyperparam tuning stage 2


In [ ]:
# ============================================================================
# STAGE 2: NUMBER OF UNITS
# ============================================================================
import shutil

import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import importlib
import helpers.hyperparameter_search.hyperparam_sweep_seqidRNN as hp_sweep
importlib.reload(hp_sweep)

from helpers.hyperparameter_search.hyperparam_sweep_seqidRNN import (
    build_configs, run_config, config_tag,
    print_results, save_results, top_configs,
    load_results
)

from helpers.rnn.seqidRNN import seqidRNN, train

# ---------------------------------------------------------------------------
# Definitions / configs
# ---------------------------------------------------------------------------

features_dir = save_path / 'rnn_features_model_identification'
results_dir_base = save_path / 'hyperparam_search' /'hyperparam_search_seqid_3stage'

STAGE_2_SETTINGS = {
    'train_size': 25000,
    'val_size': 5000,
    'epochs': 50,
    'batch_size': 32,
    'top_k': 3,
}

stage1_results = load_results(results_dir_base / "stage1")
winners = top_configs(stage1_results, k=STAGE_2_SETTINGS['top_k'])  # backbone winners (full cfg dicts)

HEAD_OPTIONS = [
    # fc0
    {"n_classifier_layers": 0, "fc_width_mult": None, "dropout": 0.2},
    # fc1
    {"n_classifier_layers": 1, "fc_width_mult": (0.5,), "dropout": (0.2, 0.1)},
    {"n_classifier_layers": 1, "fc_width_mult": (0.75,), "dropout": (0.2, 0.1)},
    {"n_classifier_layers": 1, "fc_width_mult": (1.0,), "dropout": (0.2, 0.1)},
    {"n_classifier_layers": 1, "fc_width_mult": (0.5,), "dropout": (0.1, 0.05)},
    # fc2
    {"n_classifier_layers": 2, "fc_width_mult": (0.5, 0.25), "dropout": (0.2, 0.1, 0.05)},
    {"n_classifier_layers": 2, "fc_width_mult": (0.5, 0.5), "dropout": (0.2, 0.1, 0.05)},
    {"n_classifier_layers": 2, "fc_width_mult": (0.5, 0.25), "dropout": (0.3, 0.15, 0.05)},
    {"n_classifier_layers": 2, "fc_width_mult": (0.75, 0.5), "dropout": (0.2, 0.1, 0.05)}
]
configs = []
for base in winners:
    backbone_keys = ("rnn_type", "bidirectional", "hidden_size", "lr", "patience", "reduce_lr_patience")
    backbone = {k: base[k] for k in backbone_keys if k in base}
    for head in HEAD_OPTIONS:
        cfg = {**backbone, **head}
        configs.append(cfg)
print(f"Stage 2 configs: {len(configs)} (= {len(winners)} backbones × {len(HEAD_OPTIONS)} heads)")

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

print("\nLoading data...")

X_train = np.load(features_dir / "features_train.npy", mmap_mode='r')
X_val = np.load(features_dir / "features_val.npy", mmap_mode='r')
y_train = np.load(features_dir / "labels_train.npy")
y_val = np.load(features_dir / "labels_val.npy")

print(f"  Train: {X_train.shape}, {y_train.shape}")
print(f"  Val:   {X_val.shape}, {y_val.shape}")

INPUT_SIZE = X_train.shape[-1]
N_MODELS = y_train.shape[1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Prepare data subset
# ---------------------------------------------------------------------------
print("\nSubsetting data...")
rng = np.random.RandomState(42)

train_idx = rng.choice(len(y_train), size=STAGE_2_SETTINGS['train_size'], replace=False)
val_idx = rng.choice(len(y_val), size=STAGE_2_SETTINGS['val_size'], replace=False)

sessions_train = torch.tensor(np.array(X_train[train_idx]), dtype=torch.float32)
sessions_val   = torch.tensor(np.array(X_val[val_idx]),     dtype=torch.float32)
labels_train   = torch.tensor(y_train[train_idx].argmax(axis=1), dtype=torch.long)
labels_val     = torch.tensor(y_val[val_idx].argmax(axis=1),     dtype=torch.long)

n_real_train = (sessions_train[:, :, 0, 0] != -1).sum(dim=1).long()
n_real_val   = (sessions_val[:, :, 0, 0] != -1).sum(dim=1).long()

print(f"  Train: {sessions_train.shape}, real sessions: {n_real_train.min()}-{n_real_train.max()}")
print(f"  Val:   {sessions_val.shape}, real sessions: {n_real_val.min()}-{n_real_val.max()}")
print(f"  Device: {DEVICE}")

BS = STAGE_2_SETTINGS['batch_size']
train_loader = DataLoader(TensorDataset(sessions_train, labels_train, n_real_train), batch_size=BS, shuffle=True)
val_loader   = DataLoader(TensorDataset(sessions_val, labels_val, n_real_val), batch_size=BS)


# ---------------------------------------------------------------------------
# Run Stage 2
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print(f"STAGE 2: {len(configs)} configs (top {STAGE_2_SETTINGS['top_k']} stage 1 architectures × {len([64,128,256])} hidden sizes)")
print("=" * 70)

SAVE_DIR_S1 = results_dir_base / "stage1"
SAVE_DIR = results_dir_base / "stage2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy stage 1 results that match stage 2 configs
stage2_tags = {config_tag(cfg) for cfg in configs}
for pt_file in SAVE_DIR_S1.glob("*.pt"):
    if pt_file.stem in stage2_tags:
        shutil.copy2(pt_file, SAVE_DIR / pt_file.name)
        print(f"  Copied {pt_file.name} from stage 1")

stage2_results = []
for i, cfg in enumerate(configs):
    print(f"\n--- Config {i+1}/{len(configs)}: {config_tag(cfg)} ---")
    result = run_config(
        cfg, train_loader, val_loader,
        INPUT_SIZE, N_MODELS, DEVICE,
        n_epochs=STAGE_2_SETTINGS['epochs'],
        save_dir=SAVE_DIR,
    )
    stage2_results.append(result)
    print(f"    val_acc={result['best_val_acc']:.4f}")

# Final ranked summary overwrites the append-order JSON
save_results(stage2_results, SAVE_DIR)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print_results(stage2_results, "STAGE 2 RESULTS")

top_stage2 = top_configs(stage2_results, k=STAGE_2_SETTINGS['top_k'])
print(f"\nCarrying top {STAGE_2_SETTINGS['top_k']} to Stage 2:")
for cfg in top_stage2:
    print(f"  {config_tag(cfg)}")

Stage 2 configs: 27 (= 3 backbones × 9 heads)

Loading data...
  Train: (80000, 35, 300, 11), (80000, 2)
  Val:   (10000, 35, 300, 11), (10000, 2)

Subsetting data...
  Train: torch.Size([25000, 35, 300, 11]), real sessions: 3-35
  Val:   torch.Size([5000, 35, 300, 11]), real sessions: 3-35
  Device: cuda

STAGE 2: 27 configs (top 3 stage 1 architectures × 3 hidden sizes)
  Copied lstm_uni_128u_fc0_d0.2.pt from stage 1
  Copied lstm_uni_128u_fc2_m0.5-0.25_d0.2-0.1-0.05.pt from stage 1
  Copied gru_uni_256u_fc1_m0.5_d0.2-0.1.pt from stage 1
  Copied lstm_uni_256u_fc1_m0.5_d0.2-0.1.pt from stage 1
  Copied lstm_uni_128u_fc1_m0.5_d0.2-0.1.pt from stage 1

--- Config 1/27: lstm_uni_128u_fc0_d0.2 ---
    [SKIP] Found lstm_uni_128u_fc0_d0.2.pt, loading cached result
    val_acc=0.9752

--- Config 2/27: lstm_uni_128u_fc1_m0.5_d0.2-0.1 ---
    [SKIP] Found lstm_uni_128u_fc1_m0.5_d0.2-0.1.pt, loading cached result
    val_acc=0.9868

--- Config 3/27: lstm_uni_128u_fc1_m0.75_d0.2-0.1 ---
    [SK

Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

Early stopping at epoch 39
    val_acc=0.9784

STAGE 2 RESULTS
  1. lstm_uni_128u_fc1_m0.5_d0.2-0.1  val_acc=0.9868  epoch=50/50 (max_epochs)
  2. lstm_uni_256u_fc2_m0.75-0.5_d0.2-0.1-0.05  val_acc=0.9832  epoch=41/50 (max_epochs)
  3. lstm_uni_256u_fc1_m0.75_d0.2-0.1  val_acc=0.9858  epoch=44/50 (max_epochs)
  4. gru_uni_256u_fc2_m0.5-0.25_d0.2-0.1-0.05  val_acc=0.9812  epoch=36/50 (max_epochs)
  5. lstm_uni_256u_fc0_d0.2  val_acc=0.9814  epoch=29/44 (early_stopping)
  6. lstm_uni_128u_fc1_m1_d0.2-0.1  val_acc=0.9836  epoch=44/50 (max_epochs)
  7. gru_uni_256u_fc2_m0.5-0.5_d0.2-0.1-0.05  val_acc=0.9814  epoch=22/37 (early_stopping)
  8. lstm_uni_256u_fc1_m0.5_d0.2-0.1  val_acc=0.9838  epoch=40/50 (max_epochs)
  9. gru_uni_256u_fc1_m1_d0.2-0.1  val_acc=0.9798  epoch=28/43 (early_stopping)
  10. lstm_uni_128u_fc1_m0.75_d0.2-0.1  val_acc=0.9822  epoch=45/50 (max_epochs)
  11. gru_uni_256u_fc1_m0.5_d0.1-0.05  val_acc=0.9788  epoch=21/36 (early_stopping)
  12. gru_uni_256u_fc0_d0.2  val_ac

In [ ]:
# ============================================================================
# STAGE 2: NUMBER OF UNITS
# ============================================================================
import shutil

import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import importlib
import helpers.hyperparameter_search.hyperparam_sweep_seqidRNN as hp_sweep
importlib.reload(hp_sweep)

from helpers.hyperparameter_search.hyperparam_sweep_seqidRNN import (
    build_configs, run_config, config_tag,
    print_results, save_results, top_configs,
    load_results
)

from helpers.rnn.seqidRNN import seqidRNN, train

# ---------------------------------------------------------------------------
# Definitions / configs
# ---------------------------------------------------------------------------

features_dir = save_path / 'rnn_features_model_identification'
results_dir_base = save_path / 'hyperparam_search' /'hyperparam_search_seqid_3stage'

STAGE_2_SETTINGS = {
    'train_size': 25000,
    'val_size': 5000,
    'epochs': 50,
    'batch_size': 32,
    'top_k': 3,
}

TOP_K = 3

# get winning models
stage1_results = load_results(results_dir_base / "stage1")
print_results(stage1_results, "STAGE 1 RESULTS (loaded)")
# Remove keys that stage 2 will sweep over
winners = [{k: v for k, v in cfg.items() if k != 'hidden_size'} for cfg in top_configs(stage1_results, k=TOP_K)]

HYPSEARCH_CONFIG_STAGE2 = {
    'hidden_size': [64, 128, 256],
    '_paired': winners,  # each winner config becomes a paired option
    # Override training settings
    'lr': 1e-3,
    'patience': 15,
    'reduce_lr_patience': 7,
}

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

print("\nLoading data...")

X_train = np.load(features_dir / "features_train.npy", mmap_mode='r')
X_val = np.load(features_dir / "features_val.npy", mmap_mode='r')
y_train = np.load(features_dir / "labels_train.npy")
y_val = np.load(features_dir / "labels_val.npy")

print(f"  Train: {X_train.shape}, {y_train.shape}")
print(f"  Val:   {X_val.shape}, {y_val.shape}")

INPUT_SIZE = X_train.shape[-1]
N_MODELS = y_train.shape[1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Prepare data subset
# ---------------------------------------------------------------------------
print("\nSubsetting data...")
rng = np.random.RandomState(42)

train_idx = rng.choice(len(y_train), size=STAGE_2_SETTINGS['train_size'], replace=False)
val_idx = rng.choice(len(y_val), size=STAGE_2_SETTINGS['val_size'], replace=False)

sessions_train = torch.tensor(np.array(X_train[train_idx]), dtype=torch.float32)
sessions_val   = torch.tensor(np.array(X_val[val_idx]),     dtype=torch.float32)
labels_train   = torch.tensor(y_train[train_idx].argmax(axis=1), dtype=torch.long)
labels_val     = torch.tensor(y_val[val_idx].argmax(axis=1),     dtype=torch.long)

n_real_train = (sessions_train[:, :, 0, 0] != -1).sum(dim=1).long()
n_real_val   = (sessions_val[:, :, 0, 0] != -1).sum(dim=1).long()

print(f"  Train: {sessions_train.shape}, real sessions: {n_real_train.min()}-{n_real_train.max()}")
print(f"  Val:   {sessions_val.shape}, real sessions: {n_real_val.min()}-{n_real_val.max()}")
print(f"  Device: {DEVICE}")

BS = STAGE_2_SETTINGS['batch_size']
train_loader = DataLoader(TensorDataset(sessions_train, labels_train, n_real_train), batch_size=BS, shuffle=True)
val_loader   = DataLoader(TensorDataset(sessions_val, labels_val, n_real_val), batch_size=BS)


# ---------------------------------------------------------------------------
# Run Stage 2
# ---------------------------------------------------------------------------

configs = build_configs(HYPSEARCH_CONFIG_STAGE2)

print("\n" + "=" * 70)
print(f"STAGE 2: {len(configs)} configs (top {TOP_K} stage 1 architectures × {len([64,128,256])} hidden sizes)")
print("=" * 70)

SAVE_DIR_S1 = results_dir_base / "stage1"
SAVE_DIR = results_dir_base / "stage2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy stage 1 results that match stage 2 configs
stage2_tags = {config_tag(cfg) for cfg in configs}
for pt_file in SAVE_DIR_S1.glob("*.pt"):
    if pt_file.stem in stage2_tags:
        shutil.copy2(pt_file, SAVE_DIR / pt_file.name)
        print(f"  Copied {pt_file.name} from stage 1")

stage2_results = []
for i, cfg in enumerate(configs):
    print(f"\n--- Config {i+1}/{len(configs)}: {config_tag(cfg)} ---")
    result = run_config(
        cfg, train_loader, val_loader,
        INPUT_SIZE, N_MODELS, DEVICE,
        n_epochs=STAGE_2_SETTINGS['epochs'],
        save_dir=SAVE_DIR,
    )
    stage2_results.append(result)
    print(f"    val_acc={result['best_val_acc']:.4f}")

# Final ranked summary overwrites the append-order JSON
save_results(stage2_results, SAVE_DIR)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print_results(stage2_results, "STAGE 2 RESULTS")

top_stage2 = top_configs(stage2_results, k=STAGE_2_SETTINGS['top_k'])
print(f"\nCarrying top {STAGE_2_SETTINGS['top_k']} to Stage 2:")
for cfg in top_stage2:
    print(f"  {config_tag(cfg)}")


STAGE 1 RESULTS (loaded)
  1. lstm_uni_128u_fc1_m0.5_d0.2-0.1  val_acc=0.9868  epoch=50/50 (max_epochs)
  2. lstm_uni_256u_fc1_m0.5_d0.2-0.1  val_acc=0.9838  epoch=40/50 (max_epochs)
  3. gru_uni_256u_fc1_m0.5_d0.2-0.1  val_acc=0.9800  epoch=28/43 (early_stopping)
  4. gru_uni_128u_fc1_m0.5_d0.2-0.1  val_acc=0.9774  epoch=29/44 (early_stopping)
  5. gru_uni_64u_fc1_m0.5_d0.2-0.1  val_acc=0.9726  epoch=46/50 (max_epochs)
  6. lstm_uni_64u_fc1_m0.5_d0.2-0.1  val_acc=0.9414  epoch=45/50 (max_epochs)
  7. gru_bi_128u_fc1_m0.5_d0.2-0.1  val_acc=0.8736  epoch=24/39 (early_stopping)
  8. gru_bi_256u_fc1_m0.5_d0.2-0.1  val_acc=0.8610  epoch=11/26 (early_stopping)
  9. lstm_bi_256u_fc1_m0.5_d0.2-0.1  val_acc=0.8368  epoch=29/44 (early_stopping)
  10. gru_bi_64u_fc1_m0.5_d0.2-0.1  val_acc=0.8236  epoch=21/36 (early_stopping)
  11. lstm_bi_128u_fc1_m0.5_d0.2-0.1  val_acc=0.8048  epoch=34/49 (early_stopping)
  12. lstm_bi_64u_fc1_m0.5_d0.2-0.1  val_acc=0.8046  epoch=40/50 (max_epochs)

Loading da

### Hyperparam tuning stage 3

In [ ]:
# ============================================================================
# STAGE 3: DROPOUT SWEEP
# ============================================================================
import shutil
import itertools

import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import importlib
import helpers.hyperparameter_search.hyperparam_sweep_seqidRNN as hp_sweep
importlib.reload(hp_sweep)

from helpers.hyperparameter_search.hyperparam_sweep_seqidRNN import (
    build_configs, run_config, config_tag,
    print_results, save_results, top_configs,
    load_results,
)

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
features_dir = save_path / 'rnn_features_model_identification'
results_dir_base = save_path / 'hyperparam_search' /'hyperparam_search_seqid_3stage'

STAGE_3_SETTINGS = {
    'train_size': 25000,
    'val_size': 5000,
    'epochs': 50,
    'batch_size': 32,
    'top_k': 3,
}

TOP_K = 3
DROPOUT_OPTIONS = [0.05, 0.1, 0.2]

# ---------------------------------------------------------------------------
# Load stage 2 winners
# ---------------------------------------------------------------------------
stage2_results = load_results(results_dir_base / "stage2")
print_results(stage2_results, "STAGE 2 RESULTS (loaded)")

winners = top_configs(stage2_results, k=TOP_K)

# Build dropout combos per winner (monotonically decreasing only)
stage3_paired = []
for cfg in winners:
    n_cls = cfg.get("n_classifier_layers", 0)
    n_dropouts = n_cls + 1

    # All monotonically decreasing dropout combos
    dropout_combos = [
        d for d in itertools.product(DROPOUT_OPTIONS, repeat=n_dropouts)
        if all(d[i] >= d[i + 1] for i in range(len(d) - 1))
    ]

    # Strip dropout from winner config, add each combo
    base = {k: v for k, v in cfg.items() if k != 'dropout'}
    for d_combo in dropout_combos:
        stage3_paired.append({**base, 'dropout': d_combo if n_dropouts > 1 else d_combo[0]})

print(f"\n{len(stage3_paired)} dropout configs across {TOP_K} architectures")

HYPSEARCH_CONFIG_STAGE3 = {
    '_paired': stage3_paired,
    'lr': 1e-3,
    'patience': 15,
    'reduce_lr_patience': 7,
}

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
print("\nLoading data...")

X_train = np.load(features_dir / "features_train.npy", mmap_mode='r')
X_val = np.load(features_dir / "features_val.npy", mmap_mode='r')
y_train = np.load(features_dir / "labels_train.npy")
y_val = np.load(features_dir / "labels_val.npy")

INPUT_SIZE = X_train.shape[-1]
N_MODELS = y_train.shape[1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nSubsetting data...")
rng = np.random.RandomState(42)
train_idx = rng.choice(len(y_train), size=STAGE_3_SETTINGS['train_size'], replace=False)
val_idx = rng.choice(len(y_val), size=STAGE_3_SETTINGS['val_size'], replace=False)

sessions_train = torch.tensor(np.array(X_train[train_idx]), dtype=torch.float32)
sessions_val   = torch.tensor(np.array(X_val[val_idx]),     dtype=torch.float32)
labels_train   = torch.tensor(y_train[train_idx].argmax(axis=1), dtype=torch.long)
labels_val     = torch.tensor(y_val[val_idx].argmax(axis=1),     dtype=torch.long)

n_real_train = (sessions_train[:, :, 0, 0] != -1).sum(dim=1).long()
n_real_val   = (sessions_val[:, :, 0, 0] != -1).sum(dim=1).long()

print(f"  Train: {sessions_train.shape}, real sessions: {n_real_train.min()}-{n_real_train.max()}")
print(f"  Val:   {sessions_val.shape}, real sessions: {n_real_val.min()}-{n_real_val.max()}")
print(f"  Device: {DEVICE}")

BS = STAGE_3_SETTINGS['batch_size']
train_loader = DataLoader(TensorDataset(sessions_train, labels_train, n_real_train), batch_size=BS, shuffle=True)
val_loader   = DataLoader(TensorDataset(sessions_val, labels_val, n_real_val), batch_size=BS)

# ---------------------------------------------------------------------------
# Run Stage 3
# ---------------------------------------------------------------------------
configs = build_configs(HYPSEARCH_CONFIG_STAGE3)

print("\n" + "=" * 70)
print(f"STAGE 3: DROPOUT SWEEP — {len(configs)} configs")
print("=" * 70)

SAVE_DIR = results_dir_base / "stage3"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy matching results from stage 1 and stage 2
stage3_tags = {config_tag(cfg) for cfg in configs}
for prev_dir in [results_dir_base / "stage1", results_dir_base / "stage2"]:
    for pt_file in prev_dir.glob("*.pt"):
        if pt_file.stem in stage3_tags and not (SAVE_DIR / pt_file.name).exists():
            shutil.copy2(pt_file, SAVE_DIR / pt_file.name)
            print(f"  Copied {pt_file.name} from {prev_dir.name}")

stage3_results = []
for i, cfg in enumerate(configs):
    print(f"\n--- Config {i+1}/{len(configs)}: {config_tag(cfg)} ---")
    result = run_config(
        cfg, train_loader, val_loader,
        INPUT_SIZE, N_MODELS, DEVICE,
        n_epochs=STAGE_3_SETTINGS['epochs'],
        save_dir=SAVE_DIR,
    )
    stage3_results.append(result)
    print(f"    val_acc={result['best_val_acc']:.4f}")

save_results(stage3_results, SAVE_DIR)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print_results(stage3_results, "STAGE 3 RESULTS")

top_stage3 = top_configs(stage3_results, k=STAGE_3_SETTINGS['top_k'])
print(f"\nTop {STAGE_3_SETTINGS['top_k']} configs:")
for cfg in top_stage3:
    print(f"  {config_tag(cfg)}")


STAGE 2 RESULTS (loaded)
  1. lstm_uni_256u_fc1_d0.2-0.1  val_acc=0.9838  epoch=40/50 (max_epochs)
  2. gru_uni_256u_fc1_d0.2-0.1  val_acc=0.9800  epoch=28/43 (early_stopping)
  3. gru_uni_64u_fc1_d0.2-0.1  val_acc=0.9726  epoch=46/50 (max_epochs)
  4. lstm_uni_64u_fc0_d0.2  val_acc=0.9572  epoch=48/50 (max_epochs)
  5. lstm_uni_64u_fc1_d0.2-0.1  val_acc=0.9414  epoch=45/50 (max_epochs)

18 dropout configs across 3 architectures

Loading data...

Subsetting data...
  Train: torch.Size([25000, 35, 300, 11]), real sessions: 3-35
  Val:   torch.Size([5000, 35, 300, 11]), real sessions: 3-35
  Device: cuda

STAGE 3: DROPOUT SWEEP — 18 configs
  Copied gru_uni_64u_fc1_d0.2-0.1.pt from stage2
  Copied lstm_uni_256u_fc1_d0.2-0.1.pt from stage2
  Copied gru_uni_256u_fc1_d0.2-0.1.pt from stage2

--- Config 1/18: lstm_uni_256u_fc1_d0.05-0.05 ---


Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

# Part 4: Fine-tuning

In [ ]:
# ============================================================================
# FINAL TRAINING: Best architecture on full data (load if already finetuned)
# ============================================================================

import torch
from torch.utils.data import DataLoader

import importlib
import helpers.hyperparameter_search.hyperparam_sweep_seqidRNN as hp_sweep
importlib.reload(hp_sweep)
from helpers.hyperparameter_search.hyperparam_sweep_seqidRNN import load_results, top_configs, config_tag

from helpers.rnn.seqidRNN import seqidRNN, train, evaluate
from helpers.rnn.datasets import SessionNpyDataset

# Same tree as stage 1–3 cells; finetuned checkpoints live in .../finetuned/
results_dir_base = save_path / "hyperparam_search_seqid_3stage"
features_dir = save_path / "rnn_features_model_identification"
final_dir = results_dir_base / "finetuned"
final_dir.mkdir(parents=True, exist_ok=True)

stage3_results = load_results(results_dir_base / "stage3")
best_cfg = dict(top_configs(stage3_results, k=1)[0])
tag = config_tag(best_cfg)
final_pt = final_dir / f"final_{tag}.pt"

print(f"Best config tag: {tag}")
print(f"Checkpoint path: {final_pt}")

# Longer / full-data training; batch size can stay at sweep value or increase if memory allows
FINAL_SETTINGS = {
    "epochs": 200,
    "batch_size": 32,
    "lr": best_cfg.get("lr", 1e-3),
    "patience": max(best_cfg.get("patience", 15), 20),
    "reduce_lr_patience": best_cfg.get("reduce_lr_patience", 7),
}

print("\nOpening feature/label arrays (memmap)...")
train_ds = SessionNpyDataset(features_dir / "features_train.npy", features_dir / "labels_train.npy")
val_ds = SessionNpyDataset(features_dir / "features_val.npy", features_dir / "labels_val.npy")
test_ds = SessionNpyDataset(features_dir / "features_test.npy", features_dir / "labels_test.npy")

print(f"  Train: {train_ds.X.shape}")
print(f"  Val:   {val_ds.X.shape}")
print(f"  Test:  {test_ds.X.shape}")

INPUT_SIZE = train_ds.X.shape[-1]
N_MODELS = train_ds.y.shape[1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {DEVICE}")

BS = FINAL_SETTINGS["batch_size"]
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)

model = seqidRNN(
    input_size=INPUT_SIZE,
    hidden_size=best_cfg.get("hidden_size", 64),
    n_models=N_MODELS,
    cell_type=best_cfg.get("cell_type") or best_cfg.get("rnn_type", "gru"),
    bidirectional=best_cfg.get("bidirectional", False),
    dropout=best_cfg.get("dropout", 0.2),
    n_classifier_layers=best_cfg.get("n_classifier_layers", 0),
    fc_width_mult=best_cfg.get("fc_width_mult", 0.5),
)
model.to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

if final_pt.exists():
    print("\n" + "=" * 70)
    print(f"LOADING FINETUNED MODEL: {final_pt.name}")
    print("=" * 70)
    ckpt = torch.load(final_pt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    history = ckpt.get("history", None)
else:
    print("\n" + "=" * 70)
    print(f"FINAL TRAINING: {tag}")
    print(f"  {len(train_ds)} train / {len(val_ds)} val / {len(test_ds)} test")
    print(f"  {FINAL_SETTINGS['epochs']} max epochs, patience={FINAL_SETTINGS['patience']}")
    print("=" * 70)

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        n_epochs=FINAL_SETTINGS["epochs"],
        lr=FINAL_SETTINGS["lr"],
        patience=FINAL_SETTINGS["patience"],
        reduce_lr_patience=FINAL_SETTINGS["reduce_lr_patience"],
    )

print("\n" + "=" * 70)
print("EVALUATION")
print("=" * 70)

test_metrics = evaluate(model, test_loader, DEVICE)
train_metrics = evaluate(model, train_loader, DEVICE)
val_metrics = evaluate(model, val_loader, DEVICE)

print(f"  Train acc: {train_metrics['accuracy']:.4f}  loss: {train_metrics['loss']:.4f}")
print(f"  Val acc:   {val_metrics['accuracy']:.4f}  loss: {val_metrics['loss']:.4f}")
print(f"  Test acc:  {test_metrics['accuracy']:.4f}  loss: {test_metrics['loss']:.4f}")
if isinstance(history, dict) and "best_epoch" in history:
    print(f"  Best epoch: {history['best_epoch']}/{history['n_epochs_trained']} ({history['stop_reason']})")

if not final_pt.exists():
    torch.save(
        {
            "config": best_cfg,
            "model_state": model.state_dict(),
            "history": history,
            "test_accuracy": test_metrics["accuracy"],
            "test_loss": test_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "train_accuracy": train_metrics["accuracy"],
        },
        final_pt,
    )
    print(f"\nSaved to {final_pt}")

Best config tag: lstm_uni_256u_fc1_m0.5_d0.2-0.1
Checkpoint path: /content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/hyperparam_search_seqid_3stage/finetuned/final_lstm_uni_256u_fc1_m0.5_d0.2-0.1.pt

Opening feature/label arrays (memmap)...
  Train: (80000, 35, 300, 11)
  Val:   (10000, 35, 300, 11)
  Test:  (10000, 35, 300, 11)
  Device: cuda
  Parameters: 308,610

FINAL TRAINING: lstm_uni_256u_fc1_m0.5_d0.2-0.1
  80000 train / 10000 val / 10000 test
  200 max epochs, patience=20


Training:   0%|          | 0/200 [00:00<?, ?epoch/s]

Early stopping at epoch 41

EVALUATION
  Train acc: 0.9847  loss: 0.0477
  Val acc:   0.9784  loss: 0.0706
  Test acc:  0.9791  loss: 0.0642
  Best epoch: 21/41 (early_stopping)

Saved to /content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/hyperparam_search_seqid_3stage/finetuned/final_lstm_uni_256u_fc1_m0.5_d0.2-0.1.pt


###

# Part 5: Inspect classes

todo:
- look at confusion matrix (broken down by model version)
- look at param values for correctly vs incorrectly predicted

### 2x2 confusion matrix

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import helpers
import helpers.rnn.datasets
importlib.reload(helpers.rnn.datasets)

from helpers import io
from helpers.rnn.seqidRNN import seqidRNN, evaluate, gather_predictions
from helpers.rnn.datasets import SessionNpyDataset

# --- paths ---
if "data_path" not in globals():
    data_path = save_path.parent / "data"  # storage/data when save_path is storage/model_identification

features_dir = save_path / "rnn_features_model_identification"
final_dir = save_path / "hyperparam_search_seqid_3stage" / "finetuned"

CATEGORY_NAMES = list(MODEL_NAMES)


def _family_idx(name: str) -> int:
    s = str(name)
    if "__" in s:
        s = s.split("__", 1)[0]
    if s.startswith("Qlearn"):
        return 0
    if s.startswith("ruleQhyp"):
        return 1
    raise ValueError(f"Cannot map family for label {name!r}")


fam_names = (str(MODEL_NAMES[0]), str(MODEL_NAMES[1]))

# --- load checkpoint + data ---
candidates = sorted(final_dir.glob("final_*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    raise FileNotFoundError(f"No final_*.pt under {final_dir}")
final_pt = candidates[0]
print(f"Using checkpoint: {final_pt}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(final_pt, map_location=DEVICE, weights_only=False)
cfg = ckpt["config"]
print("config from ckpt:", cfg)

test_ds = SessionNpyDataset(
    features_dir / "features_test.npy",
    features_dir / "labels_test.npy",
)
INPUT_SIZE = int(test_ds.X.shape[-1])
N_MODELS = int(test_ds.y.shape[1])
BS = 32
test_loader = DataLoader(
    test_ds, batch_size=BS, shuffle=False, num_workers=0, pin_memory=True
)

model = seqidRNN(
    input_size=INPUT_SIZE,
    hidden_size=cfg.get("hidden_size", 64),
    n_models=N_MODELS,
    cell_type=cfg.get("cell_type") or cfg.get("rnn_type", "gru"),
    bidirectional=cfg.get("bidirectional", False),
    dropout=cfg.get("dropout", 0.2),
    n_classifier_layers=cfg.get("n_classifier_layers", 0),
    fc_width_mult=cfg.get("fc_width_mult", 0.5),
)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()
print("Sanity check evaluate:", evaluate(model, test_loader, DEVICE))

out = gather_predictions(model, test_loader, DEVICE, return_logits=False)
y_t, y_p = out["y_true"], out["y_pred"]

# ----- 1) 2×2 family confusion (from training labels) -----
ft = np.array([_family_idx(CATEGORY_NAMES[i]) for i in y_t], dtype=np.int64)
fp = np.array([_family_idx(CATEGORY_NAMES[i]) for i in y_p], dtype=np.int64)

cm2 = confusion_matrix(ft, fp, labels=[0, 1])
row_sum2 = cm2.sum(axis=1, keepdims=True)
cm2_pct = np.divide(cm2, np.maximum(row_sum2, 1), dtype=float) * 100.0

fig1, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm2_pct, interpolation="nearest", cmap="Blues", vmin=0, vmax=100)
fig1.colorbar(im, ax=ax, label="Percent of true class")
ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=fam_names,
    yticklabels=fam_names,
    xlabel="Predicted family",
    ylabel="True family",
    title="2×2 family confusion (test), % of true class",
)
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, f"{cm2_pct[i, j]:.1f}%", ha="center", va="center", color="black"
        )
plt.tight_layout()
plt.show()

# ----- 2) N×2: true simulator variant (metadata) × predicted family -----
META_RULEQHYP = io.load_pickle(data_path / "ruleQhyp" / "sim_metadata.pkl")
META_QLEARN = io.load_pickle(data_path / "Qlearn" / "sim_metadata.pkl")

with open(features_dir / "split_info.pkl", "rb") as f:
    split_info = pickle.load(f)
idx_test = np.asarray(split_info["split_indices"]["test"], dtype=np.int64)

missing = "__missing_model_id__"


def _stacked_variant_labels(meta_rq, meta_ql, key="model_id"):
    parts = []
    for namespace, meta in (("ruleQhyp", meta_rq), ("Qlearn", meta_ql)):
        for aid in sorted(meta.keys()):
            mid = str(meta[aid].get(key, missing))
            parts.append(f"{namespace}||{mid}")
    return np.asarray(parts, dtype=object)


variant_global = _stacked_variant_labels(META_RULEQHYP, META_QLEARN)
variant_test = variant_global[idx_test]

if len(variant_test) != len(y_p):
    raise ValueError(
        f"metadata/test mismatch: {len(variant_test)} variant rows vs {len(y_p)} predictions"
    )

fp2 = np.array(
    [_family_idx(MODEL_NAMES[int(j)]) for j in y_p],
    dtype=np.int64,
)

row_labels = sorted(np.unique(variant_test).tolist())
Nvar = len(row_labels)
ridx = {lab: i for i, lab in enumerate(row_labels)}

nx2 = np.zeros((Nvar, 2), dtype=np.int64)
for v_str, pf in zip(variant_test, fp2):
    nx2[ridx[str(v_str)], int(pf)] += 1

row_sum = nx2.sum(axis=1, keepdims=True)
nx2_pct = np.divide(nx2, np.maximum(row_sum, 1), dtype=float) * 100.0

fig_h = max(6.0, 0.16 * Nvar)
fig2, ax = plt.subplots(figsize=(7, fig_h))
im2 = ax.imshow(
    nx2_pct,
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=100,
    interpolation="nearest",
)
fig2.colorbar(im2, ax=ax, label="Percent of true variant (row)")
ax.set_xticks([0, 1])
ax.set_xticklabels(fam_names)
ax.set_yticks(np.arange(Nvar))
ax.set_yticklabels(row_labels, fontsize=max(5, 9 - Nvar // 25))
ax.set_xlabel("Predicted family (2-way classifier)")
ax.set_ylabel("True variant (metadata + split)")
ax.set_title("N variants × 2 predicted families (test), % of row")

for i in range(Nvar):
    for j in range(2):
        ax.text(
            j,
            i,
            f"{nx2_pct[i, j]:.0f}%",
            ha="center",
            va="center",
            fontsize=7,
            color="black",
        )
plt.tight_layout()
plt.show()

print(f"N variants in test split: {Nvar}")

Output hidden; open in https://colab.research.google.com to view.

# Part 6: Model Identification

In [ ]:
# ============================================================================
# INFERENCE: Load finetuned seqidRNN and run on real data (features_real.npy)
# ============================================================================
# Requires: save_path (Path), MODEL_NAMES e.g. ["Qlearn", "ruleQhyp"]

import importlib
import sys
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import helpers.rnn.seqidRNN  # registers helpers.rnn.seqidRNN
importlib.reload(sys.modules["helpers.rnn.seqidRNN"])

from helpers.rnn.seqidRNN import seqidRNN, _session_loop

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {DEVICE}")
BATCH = 64
MASK_VALUE = -1.0

results_dir_base = save_path / "hyperparam_search_seqid_3stage"
final_dir = results_dir_base / "finetuned"

candidates = sorted(final_dir.glob("final_*.pt"))
if not candidates:
    raise FileNotFoundError(f"No finetuned checkpoints found in {final_dir}")

best_final_pt = None
best_score = -1.0
for p in candidates:
    ck = torch.load(p, map_location="cpu", weights_only=False)
    score = float(ck.get("test_accuracy", ck.get("val_accuracy", float("nan"))))
    if score == score and score > best_score:
        best_score = score
        best_final_pt = p

if best_final_pt is None:
    best_final_pt = max(candidates, key=lambda p: p.stat().st_mtime)
    best_score = float("nan")

final_pt = best_final_pt
print(f"Using finetuned checkpoint: {final_pt.name} (score={best_score})")

real_npy_dir = save_path / "real_features_model_identification"
features_real_path = real_npy_dir / "features_real.npy"
subject_ids_path = real_npy_dir / "subject_ids_real.npy"

ckpt = torch.load(final_pt, map_location="cpu", weights_only=False)
cfg = dict(ckpt["config"])

X_real = np.load(features_real_path, mmap_mode="r")
assert X_real.ndim == 4 and X_real.shape[-1] == 11, X_real.shape

X_real_w = np.array(X_real, dtype=np.float32, copy=True)
sessions_real = torch.from_numpy(X_real_w)

# Non-pad session slab: at least one trial has at least one feature != MASK_VALUE
is_pad_sess = (sessions_real == MASK_VALUE).all(dim=-1).all(dim=-1)
n_real_sessions = (~is_pad_sess).sum(dim=1).long()

if subject_ids_path.exists():
    subject_ids = [str(x) for x in np.load(subject_ids_path, allow_pickle=True).tolist()]
else:
    subject_ids = [str(i) for i in range(sessions_real.shape[0])]

if len(subject_ids) != sessions_real.shape[0]:
    raise ValueError(
        f"len(subject_ids)={len(subject_ids)} != N={sessions_real.shape[0]}"
    )

dummy_y = torch.zeros(sessions_real.shape[0], dtype=torch.long)

loader = DataLoader(
    TensorDataset(sessions_real, dummy_y, n_real_sessions),
    batch_size=BATCH,
    shuffle=False,
)

INPUT_SIZE = int(X_real.shape[-1])
N_MODELS = len(MODEL_NAMES)

state = ckpt.get("model_state_by_loss") or ckpt.get("model_state")
if state is None:
    raise KeyError("Checkpoint missing model_state / model_state_by_loss")

model = seqidRNN(
    input_size=INPUT_SIZE,
    hidden_size=cfg.get("hidden_size", 64),
    n_models=N_MODELS,
    cell_type=cfg.get("cell_type") or cfg.get("rnn_type", "gru"),
    bidirectional=cfg.get("bidirectional", False),
    dropout=cfg.get("dropout", 0.2),
    n_classifier_layers=cfg.get("n_classifier_layers", 0),
    fc_width_mult=cfg.get("fc_width_mult", 0.5),
    trial_pad_value=MASK_VALUE,
)
model.load_state_dict(state, strict=True)
model.to(DEVICE)
model.eval()

all_logits = []
with torch.no_grad():
    for batch in loader:
        sess = batch[0].to(DEVICE)
        labs = batch[1].to(DEVICE)
        n_sess = batch[2].to(DEVICE)
        final_logits, _ = _session_loop(
            model, sess, labs, n_sess, compute_loss=False
        )
        all_logits.append(final_logits.cpu())

logits_real = torch.cat(all_logits, dim=0)
probs_real = F.softmax(logits_real, dim=-1).numpy()
pred_idx = logits_real.argmax(dim=-1).numpy()

print(f"Real tensor: {tuple(X_real.shape)}")
print(f"Logits: {tuple(logits_real.shape)}")
for k in range(N_MODELS):
    print(f"  P({MODEL_NAMES[k]})  mean={probs_real[:, k].mean():.4f}")

out_csv = save_path / "real_model_id_predictions.csv"
pd.DataFrame(
    {
        "subject_id": subject_ids,
        "pred_idx": pred_idx,
        "pred_name": [MODEL_NAMES[i] for i in pred_idx],
        "P_Qlearn": probs_real[:, 0],
        "P_ruleQhyp": probs_real[:, 1],
    }
).to_csv(out_csv, index=False)
print(f"Saved {out_csv}")

  Device: cuda
Using finetuned checkpoint: final_lstm_uni_256u_fc1_m0.5_d0.2-0.1.pt (score=0.9791)
Real tensor: (29, 35, 300, 11)
Logits: (29, 2)
  P(Qlearn)  mean=0.0315
  P(ruleQhyp)  mean=0.9685
Saved /content/drive/My Drive/Colab Notebooks/rule_learning_sbi/storage/model_identification/real_model_id_predictions.csv


In [ ]:
# --- Sanity: packed (-1 skip) vs legacy (feed all timesteps) ---
import sys
import importlib
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

import helpers.rnn.seqidRNN
importlib.reload(sys.modules["helpers.rnn.seqidRNN"])
from helpers.rnn.seqidRNN import seqidRNN, _session_loop

# ---------------- paths ----------------
save_path = Path("/content/drive/MyDrive/Colab Notebooks/rule_learning_sbi/storage/model_identification")  # edit
CKPT = save_path / "hyperparam_search_seqid_3stage" / "finetuned" / "final_lstm_uni_128u_fc1_m0.5_d0.2-0.1.pt"  # edit

FEATURES_TEST = save_path / "rnn_features_model_identification" / "features_test.npy"  # edit
LABELS_TEST = save_path / "rnn_features_model_identification" / "labels_test.npy"
FEATURES_REAL = save_path / "real_features_model_identification" / "features_real.npy"

MODEL_NAMES = ["Qlearn", "ruleQhyp"]
MASK_VALUE = -1.0
BATCH = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _n_real_sessions(x: torch.Tensor) -> torch.Tensor:
    is_pad = (x == MASK_VALUE).all(dim=-1).all(dim=-1)
    return (~is_pad).sum(dim=1).long()


def _load_labels(path: Path) -> torch.Tensor:
    y = np.load(path)
    if y.ndim == 2:
        y = y.argmax(axis=1)
    return torch.from_numpy(y.astype(np.int64))


@torch.no_grad()
def run_split(model, feat_np: np.ndarray, y: torch.Tensor, name: str):
    x = torch.from_numpy(np.array(feat_np, dtype=np.float32, copy=True))
    nrs = _n_real_sessions(x)
    ds = TensorDataset(x, y, nrs)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False)
    logits_all, correct, total = [], 0, 0
    for sess, lab, nr in loader:
        sess, lab, nr = sess.to(DEVICE), lab.to(DEVICE), nr.to(DEVICE)
        logits, _ = _session_loop(model, sess, lab, nr, compute_loss=False)
        logits_all.append(logits.cpu())
        pred = logits.argmax(dim=-1)
        correct += (pred == lab).sum().item()
        total += lab.numel()
    logits = torch.cat(logits_all, dim=0)
    probs = F.softmax(logits, dim=-1).numpy()
    acc = correct / max(total, 1)
    print(f"{name:26s}  n={total:5d}  acc={acc:.4f}  mean P({MODEL_NAMES[0]})={probs[:,0].mean():.4f}  mean P({MODEL_NAMES[1]})={probs[:,1].mean():.4f}")


# --- checkpoint ---
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = dict(ck["config"])
state = ck.get("model_state_by_loss") or ck.get("model_state")
Fdim = int(np.load(FEATURES_TEST, mmap_mode="r").shape[-1])


def build_model(trial_pad):
    m = seqidRNN(
        input_size=Fdim,
        hidden_size=cfg.get("hidden_size", 64),
        n_models=len(MODEL_NAMES),
        cell_type=cfg.get("cell_type") or cfg.get("rnn_type", "gru"),
        bidirectional=cfg.get("bidirectional", False),
        dropout=cfg.get("dropout", 0.2),
        n_classifier_layers=cfg.get("n_classifier_layers", 0),
        fc_width_mult=cfg.get("fc_width_mult", 0.5),
        trial_pad_value=trial_pad,
    )
    m.load_state_dict(state, strict=True)
    m.to(DEVICE).eval()
    return m


y_test = _load_labels(LABELS_TEST)
X_test = np.load(FEATURES_TEST, mmap_mode="r")
assert len(y_test) == X_test.shape[0]

print("Device:", DEVICE)
print("--- SIM TEST ---")
for mode, pad in [("packed (skip pad trials)", MASK_VALUE), ("legacy (dense + pad)", None)]:
    run_split(build_model(pad), X_test, y_test, mode)

if FEATURES_REAL.is_file():
    print("--- REAL ---")
    X_real = np.load(FEATURES_REAL, mmap_mode="r")
    y_dummy = torch.zeros(X_real.shape[0], dtype=torch.long)
    for mode, pad in [("packed", MASK_VALUE), ("legacy", None)]:
        run_split(build_model(pad), X_real, y_dummy, f"real {mode}")
else:
    print("(no features_real.npy — skipped real)")

print("\nExpect: sim acc / means nearly identical between packed vs legacy if sessions are full-length.")
print("Expect: real means may differ a lot if many short sessions (pad tails).")

Device: cuda
--- SIM TEST ---
packed (skip pad trials)    n=10000  acc=0.9874  mean P(Qlearn)=0.5166  mean P(ruleQhyp)=0.4834
legacy (dense + pad)        n=10000  acc=0.9874  mean P(Qlearn)=0.5166  mean P(ruleQhyp)=0.4834
--- REAL ---
real packed                 n=   29  acc=0.0345  mean P(Qlearn)=0.0358  mean P(ruleQhyp)=0.9642
real legacy                 n=   29  acc=0.8621  mean P(Qlearn)=0.8615  mean P(ruleQhyp)=0.1385

Expect: sim acc / means nearly identical between packed vs legacy if sessions are full-length.
Expect: real means may differ a lot if many short sessions (pad tails).


- feeding latents into MLE?
- session by session model ID predictions
